In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from datasets import load_dataset
from copy import deepcopy
import numpy as np
from scipy.stats import ttest_ind
import torch
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
df = pd.read_csv('../QA_datasets_classified_qa_eval/output_csv/HotpotQA_UND_Gemini_Ragas.csv')
ragas_col = df['ragas_AA_short'].tolist()
pre_ragas = load_dataset("json", 
                        data_files="../QA_datasets_classified_qa_eval/intermediate/BASELINE_HotpotQA_UND_qa_Gemini_with_squad_scores.jsonl",
                        split="all")
assert len(pre_ragas) == len(ragas_col), "Fatal Error: length mismatch"

Generating train split: 496 examples [00:00, 22379.01 examples/s]


In [3]:
with_ragas = pre_ragas.add_column('ragas_AA_short', ragas_col)
with_ragas.to_json("./intermediate/HotpotQA_UND_Gemini_Ragas.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 33.62ba/s]


1707792

## Rewriting with GPT-4o
GPT-4o rewriting, then Gemini QA later

In [4]:
from helper_functions_qr import modification_in_batch, find_failed_rows_simple

In [5]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
model = "gpt-4o-2024-11-20"
input_file = "./intermediate/HotpotQA_UND_Gemini_Ragas.jsonl"
output_file = "./intermediate/MODIFIED_HotpotQA_UND_Gemini_Ragas.jsonl"


In [6]:
# 清空输出文件（如果存在）
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"Cleared existing output file: {output_file}")

# 处理所有样本（按批次）
question_modification = modification_in_batch(input_file, output_file, 'answer', client, model)

Cleared existing output file: ./intermediate/MODIFIED_HotpotQA_UND_Gemini_Ragas.jsonl
Total samples to process: 496
Batch size: 3


Processing batches:  32%|███▏      | 53/166 [06:07<11:31,  6.12s/it]

Error processing sample 160: Expecting ',' delimiter: line 3 column 122 (char 271)


Processing batches:  39%|███▊      | 64/166 [07:12<10:30,  6.18s/it]

Error processing sample 194: Expecting ',' delimiter: line 3 column 228 (char 327)


...

Error processing sample 235: Invalid \escape: line 3 column 65 (char 187)


Processing batches:  52%|█████▏    | 86/166 [10:20<08:29,  6.36s/it]

Error processing sample 259: Invalid \escape: line 3 column 121 (char 229)


Processing batches: 100%|██████████| 166/166 [18:16<00:00,  6.61s/it]


All batch processing completed! Total processed: 496 samples
Results saved to: ./intermediate/MODIFIED_HotpotQA_UND_Gemini_Ragas.jsonl


In [7]:
find_failed_rows_simple(input_file, output_file)

=== 查找失败的行（简单方法）===
发现 0 个失败的行:


[]

In [8]:
df_view = pd.DataFrame(question_modification)
df_view

,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA
0,"In what year was the narrator of ""Blackadder's...","In what year was Rowan Atkinson, the actor who...",[1959],[1949],The query seeks the birth year of the narrator...,0.000000,0.0,0.00
1,What is the first two words of the fifth studi...,What are the first two words of the fifth stud...,[The Hungry],[* Same Day],"The query references 'Joseph Edgar Foreman,' w...",0.000000,0.0,0.00
2,Robert Earl Holding owned an oil company that ...,"Which oil company, originally founded by Harry...",[Harry F. Sinclair],[Harry F. Sinclair],The query requires determining the original fo...,1.000000,1.0,1.00
3,In what county was Duffy Jackson born?,"In what county was Duffy Jackson, the American...",[Nassau County],"[* Washington, D.C. is not located in a coun...",The query seeks information about the birth co...,0.222222,0.0,0.00
4,When was the defending titlist of 2009–10 Biat...,What is the birth date of Ole Einar Bjørndalen...,[27 January 1974],"[August 12, 1985]",The query seeks the birth date of the 'defendi...,0.000000,0.0,0.00
...,...,...,...,...,...,...,...,...
491,Where did Otto von Bismarck and Ludwig Friedri...,Where did Otto von Bismarck and Ludwig Friedri...,[Prussia],"[* Schönhausen, Kingdom of Prussia (Otto von...",The query asks for the places of origin of two...,0.117647,0.0,0.75
492,When was the singer of Miss Emily's Picture born?,"When was John Conlee, the singer of the countr...","[August 11, 1946]","[* June 20, 1944]",The query requires identification of the singe...,0.000000,0.0,0.00
493,On what street was the hotel located where the...,On what street was the hotel located where the...,[Peachtree Street],[Ashford Avenue],The query requires identifying a specific fire...,0.000000,0.0,0.00
494,What is the original name of the place where T...,"What was the original name of the location, no...",[Fort Saint Anthony],[Fort Snelling],The query seeks the 'original name' of the loc...,0.400000,0.0,0.25


## Modified queries QA using Gemini-2.5-Flash

### Loading modified data

In [9]:
modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_HotpotQA_UND_Gemini_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

# For issue resolution for QA, comment when there is no issue
#modified_set = modified_set.remove_columns(["model_new_answer"])

#modified_set.to_json(
    #"./intermediate/MODIFIED_HotpotQA_UND_Gemini_Ragas.jsonl",
    #orient="records",
    #lines=True
#)

Generating train split: 496 examples [00:00, 59155.33 examples/s]


### Implementation

In [10]:
from helper_functions_qr import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)
client = OpenAI(
    api_key=os.environ.get('GOOGLE_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [11]:
modified_results = batch_QA_with_progress(
    modified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_new_answer",
    fill_value=["error"],
    client=client,
    model="gemini-2.5-flash",
    temperature=0.0
)

Running model_new_answer: 100%|██████████| 50/50 [22:44<00:00, 27.29s/it]


In [12]:
qa_modified = deepcopy(modified_set)
for key in modified_results:
    qa_modified = qa_modified.add_column(key, modified_results[key])

qa_modified.to_json("./intermediate/MODIFIED_HotpotQA_UND_Gemini_Ragas.jsonl", orient="records", lines=True)
df_qa_modified = pd.read_json("./intermediate/MODIFIED_HotpotQA_UND_Gemini_Ragas.jsonl", lines=True)
#df_qa_modified.to_csv('produced_files/modification_pilot_qa.csv')
df_qa_modified

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 81.93ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer
0,"In what year was the narrator of ""Blackadder's...","In what year was Rowan Atkinson, the actor who...",[1959],[1949],The query seeks the birth year of the narrator...,0.000000,0,0.00,[1955]
1,What is the first two words of the fifth studi...,What are the first two words of the fifth stud...,[The Hungry],[* Same Day],"The query references 'Joseph Edgar Foreman,' w...",0.000000,0,0.00,[The Good]
2,Robert Earl Holding owned an oil company that ...,"Which oil company, originally founded by Harry...",[Harry F. Sinclair],[Harry F. Sinclair],The query requires determining the original fo...,1.000000,1,1.00,[Sinclair Oil Corporation]
3,In what county was Duffy Jackson born?,"In what county was Duffy Jackson, the American...",[Nassau County],"[* Washington, D.C. is not located in a coun...",The query seeks information about the birth co...,0.222222,0,0.00,"[Washington, D.C.]"
4,When was the defending titlist of 2009–10 Biat...,What is the birth date of Ole Einar Bjørndalen...,[27 January 1974],"[August 12, 1985]",The query seeks the birth date of the 'defendi...,0.000000,0,0.00,"[* January 27, 1974]"
...,...,...,...,...,...,...,...,...,...
491,Where did Otto von Bismarck and Ludwig Friedri...,Where did Otto von Bismarck and Ludwig Friedri...,[Prussia],"[* Schönhausen, Kingdom of Prussia (Otto von...",The query asks for the places of origin of two...,0.117647,0,0.75,"[* Schönhausen, Prussia\n* Berlin, Prussia]"
492,When was the singer of Miss Emily's Picture born?,"When was John Conlee, the singer of the countr...","[August 11, 1946]","[* June 20, 1944]",The query requires identification of the singe...,0.000000,0,0.00,"[August 11, 1946]"
493,On what street was the hotel located where the...,On what street was the hotel located where the...,[Peachtree Street],[Ashford Avenue],The query requires identifying a specific fire...,0.000000,0,0.00,[Peachtree Street]
494,What is the original name of the place where T...,"What was the original name of the location, no...",[Fort Saint Anthony],[Fort Snelling],The query seeks the 'original name' of the loc...,0.400000,0,0.25,[* Fort St. Anthony]


## Evaluations

### Squad EM+F1

In [13]:
# Helper functions updated, RESTART!!
from helper_functions_qr import evaluate_squad_per_sample_multi_ref_pred

modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_HotpotQA_UND_Gemini_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

qa_modified = deepcopy(modified_set)

Generating train split: 496 examples [00:00, 47398.66 examples/s]


In [14]:
squad_scored_modified, modified_f1_list, modified_em_list = evaluate_squad_per_sample_multi_ref_pred(qa_modified)
squad_scored_modified.to_json("./intermediate/MODIFIED_gpt4o_HotpotQA_UND_Gemini_new_squad.jsonl", orient="records", lines=True)

df = pd.read_json("./intermediate/MODIFIED_gpt4o_HotpotQA_UND_Gemini_new_squad.jsonl", lines=True)
df

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 105.20ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1
0,"In what year was the narrator of ""Blackadder's...","In what year was Rowan Atkinson, the actor who...",[1959],[1949],The query seeks the birth year of the narrator...,0.000000,0,0.00,[1955],0,0.000000
1,What is the first two words of the fifth studi...,What are the first two words of the fifth stud...,[The Hungry],[* Same Day],"The query references 'Joseph Edgar Foreman,' w...",0.000000,0,0.00,[The Good],0,0.000000
2,Robert Earl Holding owned an oil company that ...,"Which oil company, originally founded by Harry...",[Harry F. Sinclair],[Harry F. Sinclair],The query requires determining the original fo...,1.000000,1,1.00,[Sinclair Oil Corporation],0,0.333333
3,In what county was Duffy Jackson born?,"In what county was Duffy Jackson, the American...",[Nassau County],"[* Washington, D.C. is not located in a coun...",The query seeks information about the birth co...,0.222222,0,0.00,"[Washington, D.C.]",0,0.000000
4,When was the defending titlist of 2009–10 Biat...,What is the birth date of Ole Einar Bjørndalen...,[27 January 1974],"[August 12, 1985]",The query seeks the birth date of the 'defendi...,0.000000,0,0.00,"[* January 27, 1974]",0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
491,Where did Otto von Bismarck and Ludwig Friedri...,Where did Otto von Bismarck and Ludwig Friedri...,[Prussia],"[* Schönhausen, Kingdom of Prussia (Otto von...",The query asks for the places of origin of two...,0.117647,0,0.75,"[* Schönhausen, Prussia\n* Berlin, Prussia]",0,0.400000
492,When was the singer of Miss Emily's Picture born?,"When was John Conlee, the singer of the countr...","[August 11, 1946]","[* June 20, 1944]",The query requires identification of the singe...,0.000000,0,0.00,"[August 11, 1946]",1,1.000000
493,On what street was the hotel located where the...,On what street was the hotel located where the...,[Peachtree Street],[Ashford Avenue],The query requires identifying a specific fire...,0.000000,0,0.00,[Peachtree Street],1,1.000000
494,What is the original name of the place where T...,"What was the original name of the location, no...",[Fort Saint Anthony],[Fort Snelling],The query seeks the 'original name' of the loc...,0.400000,0,0.25,[* Fort St. Anthony],0,0.666667


In [15]:
modified_mean_em = np.mean(modified_em_list)  # em_scores: EM list per sample
modified_mean_f1 = np.mean(modified_f1_list)  # f1_scores F1 list per sample
print(f"New answers after modification Exact Match (avg): {modified_mean_em * 100:.2f}")
print(f"New answers after modification F1 Score (avg): {modified_mean_f1 * 100:.2f}")

original_em_list = qa_modified['original_em']
original_f1_list = qa_modified['original_f1']

original_mean_em = np.mean(original_em_list)  # em_scores: EM list per sample
original_mean_f1 = np.mean(original_f1_list)  # f1_scores F1 list per sample
print(f"Original answers Exact Match (avg): {original_mean_em * 100:.2f}")
print(f"Original answers F1 Score (avg): {original_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(modified_f1_list, original_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(modified_em_list, original_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

New answers after modification Exact Match (avg): 35.48
New answers after modification F1 Score (avg): 50.60
Original answers Exact Match (avg): 29.03
Original answers F1 Score (avg): 41.17
F1: t=3.378, p=0.0008
EM: t=2.176, p=0.0298


### Ragas AA

In [16]:
from helper_functions_qr import answer_accuracy_modified
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

In [17]:
squad_scored_modified = load_dataset("json",
    data_files="./intermediate/MODIFIED_gpt4o_HotpotQA_UND_Gemini_new_squad.jsonl",
    split="train")
result_with_AA = await answer_accuracy_modified(squad_scored_modified, evaluator_llm)
result_with_AA.to_csv("./output_csv/MODIFIED_gpt4o_HotpotQA_UND_Gemini_all_new_scores.csv")

Generating train split: 496 examples [00:00, 45404.19 examples/s]
Calculating short answer accuracy:   0%|          | 0/496 [00:00<?, ?it/s]

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 17.73ba/s]


485287

In [18]:
original_AA = list(result_with_AA["original_AA"])
modified_AA = list(result_with_AA["new_AA"])

original_mean_AA = np.mean(original_AA)
print(f"original AA (avg): {original_mean_AA * 100:.2f}")


modified_mean_AA = np.mean(modified_AA)
print(f"modified AA (avg): {modified_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(modified_AA, original_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

original AA (avg): 45.26
modified AA (avg): 60.48
AA: t=5.176, p=0.0000


## Re-Classification

In [2]:
from helper_functions_qr import (
    batch_generate_responses_qwen3,
    get_judgments_from_responses,
    run_experiment,
    prepare_test_prompts)

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


### Loading data

In [3]:
reclassify_file = "./output_csv/MODIFIED_gpt4o_HotpotQA_UND_Gemini_all_new_scores.csv"
df_reclassify_file = pd.read_csv(reclassify_file)

### Loading model

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
Qwen3_4B = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)
# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.37s/it]


cuda


### Prepare prompts

In [5]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [6]:
test_prompts = prepare_test_prompts(df_reclassify_file, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 496
Generation complete: 496 prompts
Average prompt length: 499 bytes (~124 tokens)

Analyze the following input user query:

{"query": "In what year was Rowan Atkinson, the actor who portrays the narrator in "Blackadder's Christmas Carol," born?"}

Please provide your analysis in the following JSON format:

{"query": "In what year was Rowan Atkinson, the actor who portrays the narrator in "Blackadder's Christmas Carol," born?", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [7]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df_reclassify_file)
test_df

100%|██████████| 100/100 [1:08:03<00:00, 40.83s/it]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1,new_AA,MODIFIED_thinking,MODIFIED_model_response,MODIFIED_model_pred
0,"In what year was the narrator of ""Blackadder's...","In what year was Rowan Atkinson, the actor who...",['1959'],['1949'],The query seeks the birth year of the narrator...,0.000000,0.0,0.00,['1955'],0.0,0.000000,0.0,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""In what year was Rowan Atkinson...",fully specified
1,What is the first two words of the fifth studi...,What are the first two words of the fifth stud...,['The Hungry'],['* Same Day'],"The query references 'Joseph Edgar Foreman,' w...",0.000000,0.0,0.00,['The Good'],0.0,0.000000,0.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What are the first two words of...",fully specified
2,Robert Earl Holding owned an oil company that ...,"Which oil company, originally founded by Harry...",['Harry F. Sinclair'],['Harry F. Sinclair'],The query requires determining the original fo...,1.000000,1.0,1.00,['Sinclair Oil Corporation'],0.0,0.333333,0.5,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""Which oil company, originally f...",fully specified
3,In what county was Duffy Jackson born?,"In what county was Duffy Jackson, the American...",['Nassau County'],"['* Washington, D.C. is not located in a cou...",The query seeks information about the birth co...,0.222222,0.0,0.00,"['Washington, D.C.']",0.0,0.000000,0.0,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""In what county was Duffy Jackso...",fully specified
4,When was the defending titlist of 2009–10 Biat...,What is the birth date of Ole Einar Bjørndalen...,['27 January 1974'],"['August 12, 1985']",The query seeks the birth date of the 'defendi...,0.000000,0.0,0.00,"['* January 27, 1974']",0.0,1.000000,1.0,"<think>\nOkay, let's see. The user is asking f...","{\n ""query"": ""What is the birth date of Ole E...",fully specified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491,Where did Otto von Bismarck and Ludwig Friedri...,Where did Otto von Bismarck and Ludwig Friedri...,['Prussia'],"['* Schönhausen, Kingdom of Prussia (Otto vo...",The query asks for the places of origin of two...,0.117647,0.0,0.75,"['* Schönhausen, Prussia\n* Berlin, Prussia']",0.0,0.400000,0.0,"<think>\nOkay, let's see. The user is asking w...","{\n ""query"": ""Where did Otto von Bismarck and...",fully specified
492,When was the singer of Miss Emily's Picture born?,"When was John Conlee, the singer of the countr...","['August 11, 1946']","['* June 20, 1944']",The query requires identification of the singe...,0.000000,0.0,0.00,"['August 11, 1946']",1.0,1.000000,1.0,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""When was John Conlee, the singe...",fully specified
493,On what street was the hotel located where the...,On what street was the hotel located where the...,['Peachtree Street'],['Ashford Avenue'],The query requires identifying a specific fire...,0.000000,0.0,0.00,['Peachtree Street'],1.0,1.000000,1.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""On what street was the hotel lo...",fully specified
494,What is the original name of the place where T...,"What was the original name of the location, no...",['Fort Saint Anthony'],['Fort Snelling'],The query seeks the 'original name' of the loc...,0.400000,0.0,0.25,['* Fort St. Anthony'],0.0,0.666667,1.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What was the original name of t...",underspecified


In [8]:
test_df['MODIFIED_model_pred'].value_counts(normalize=True)


MODIFIED_model_pred
fully specified    0.721774
underspecified     0.278226
Name: proportion, dtype: float64

In [9]:
test_df['MODIFIED_model_pred'].value_counts()

MODIFIED_model_pred
fully specified    358
underspecified     138
Name: count, dtype: int64

In [10]:
test_df.to_csv('./output_csv/HotpotQA_UND_gpt4o_rewritten_reclassified.csv')